# SnowMart 出店戦略ハンズオン（AI特化版）
## Snowflake Cortex AI 機能を一気通貫で体験する

### ストーリー
あなたはコンビニチェーン「**スノーマート**」のデータチームに配属された新メンバーです。  
スノーマートは全国に **500店舗** を展開しており、来年度中に **600店舗への拡大** を計画しています。

本部からミッションが届きました：
> 「データを使って、次に出店すべきエリアを提案してほしい。  
> 売上実績、顧客の声、社内ガイドラインを全部踏まえてね。」

今日のミッションはこの問いに **Snowflake の AI 機能だけ** で答えることです。

### タイムライン
- セットアップ確認
- Scene 1: Cortex AI Functions — レビュー分析
- Scene 2: Cortex Search — 社内ナレッジ RAG
- Scene 3: Semantic View + Cortex Analyst
- Scene 4: Cortex Agent — 出店戦略を決める
- まとめ
- Scene 5: Streamlit [オプション]

### 今日体験する4つのAI機能
1. **Cortex AI Functions** — SQL一行でLLM（感情分析・要約・分類・テキスト生成）
2. **Cortex Search** — 社内文書に自然言語で質問（RAG）
3. **Semantic View + Cortex Analyst** — 自然言語でデータ分析
4. **Cortex Agent** — 複数AIツールを自律的に組み合わせて回答


## セットアップ確認

**前提**: `setup_snowmart.sql` はセッション開始前に実行済みです。  
以下のセルでデータが正しくロードされていることを確認します。


In [ ]:
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
USE SCHEMA SNOWMART_DB.SNOWMART_SCHEMA;

In [ ]:
-- 全テーブルの行数を確認
SELECT 'SNOWMART_STORES'     AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM SNOWMART_STORES
UNION ALL SELECT 'COMPETITOR_STORES',  COUNT(*) FROM COMPETITOR_STORES
UNION ALL SELECT 'AREA_MASTER',        COUNT(*) FROM AREA_MASTER
UNION ALL SELECT 'DAILY_SALES',        COUNT(*) FROM DAILY_SALES
UNION ALL SELECT 'CUSTOMER_REVIEWS',   COUNT(*) FROM CUSTOMER_REVIEWS
UNION ALL SELECT 'STORE_DOCUMENTS',    COUNT(*) FROM STORE_DOCUMENTS;
-- 期待値: SNOWMART_STORES=500 / COMPETITOR_STORES=1000 / AREA_MASTER=50
--         DAILY_SALES=約180,000 / CUSTOMER_REVIEWS=500
--         DAILY_SALES=約365,000 / STORE_DOCUMENTS=10

In [ ]:
-- 顧客レビューの例を確認（今日の分析対象）
SELECT STORE_ID, RATING, LEFT(REVIEW_TEXT, 80) AS REVIEW_PREVIEW
FROM CUSTOMER_REVIEWS
ORDER BY RATING ASC
LIMIT 5;

---
## Scene 1: Cortex AI Functions — レビュー分析


**使う関数**:  
`SNOWFLAKE.CORTEX.SENTIMENT()` / `SUMMARIZE()` / `CLASSIFY_TEXT()` / `COMPLETE()`


### Step 1-1: 感情分析（SENTIMENT）
レビューごとに感情スコア（-1: 極ネガ 〜 +1: 極ポジ）を計算します。


In [ ]:
-- レビューごとの感情スコア
SELECT
    REVIEW_ID,
    STORE_ID,
    RATING,
    LEFT(REVIEW_TEXT, 40)                              AS REVIEW_PREVIEW,
    ROUND(SNOWFLAKE.CORTEX.SENTIMENT(REVIEW_TEXT), 3)  AS SENTIMENT_SCORE
FROM CUSTOMER_REVIEWS
ORDER BY SENTIMENT_SCORE ASC
LIMIT 20;

In [ ]:
-- 店舗別の平均感情スコア（課題店舗の発見）
SELECT
    cr.STORE_ID,
    st.STORE_NAME,
    st.PREFECTURE,
    COUNT(*)                                                           AS REVIEW_COUNT,
    ROUND(AVG(cr.RATING), 1)                                          AS AVG_RATING,
    ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(cr.REVIEW_TEXT)), 3)         AS AVG_SENTIMENT
FROM CUSTOMER_REVIEWS cr
JOIN SNOWMART_STORES st ON cr.STORE_ID = st.STORE_ID
GROUP BY cr.STORE_ID, st.STORE_NAME, st.PREFECTURE
HAVING COUNT(*) >= 3
ORDER BY AVG_SENTIMENT ASC
LIMIT 10;
-- ↑ 感情スコアが最低の店舗を確認。次のステップで原因を掘り下げます。

### Step 1-2: ネガティブレビューの要約（SUMMARIZE）
低評価レビューをまとめて要約します。全部読まなくても「何が問題か」が一瞬でわかります。


In [ ]:
-- 低評価レビュー（星2以下）をまとめて要約し、日本語に翻訳
SELECT SNOWFLAKE.CORTEX.TRANSLATE(
    SNOWFLAKE.CORTEX.SUMMARIZE(
        ARRAY_TO_STRING(
            ARRAY_AGG(REVIEW_TEXT) WITHIN GROUP (ORDER BY RATING ASC),
            '\n'
        )
    ),
    'en', 'ja'
) AS NEGATIVE_REVIEW_SUMMARY
FROM CUSTOMER_REVIEWS
WHERE RATING <= 2;

### Step 1-3: 問題カテゴリの自動分類（CLASSIFY_TEXT）
レビューを課題カテゴリに自動分類します。手動タグ付け作業は不要です。


In [ ]:
-- レビューを課題カテゴリに自動分類
SELECT
    REVIEW_ID,
    STORE_ID,
    RATING,
    LEFT(REVIEW_TEXT, 50) AS REVIEW_PREVIEW,
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        REVIEW_TEXT,
        ['接客・サービス', '品揃え・商品', '立地・アクセス', '店内環境・清潔さ', '価格・レジ待ち']
    ):label::VARCHAR AS CATEGORY
FROM CUSTOMER_REVIEWS
WHERE RATING <= 2
LIMIT 30;

In [ ]:
-- カテゴリ別の件数（どの問題が最多か）
SELECT
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        REVIEW_TEXT,
        ['接客・サービス', '品揃え・商品', '立地・アクセス', '店内環境・清潔さ', '価格・レジ待ち']
    ):label::VARCHAR AS CATEGORY,
    COUNT(*) AS COUNT
FROM CUSTOMER_REVIEWS
WHERE RATING <= 2
GROUP BY CATEGORY
ORDER BY COUNT DESC;

### Step 1-4: LLMに改善施策を提案させる（COMPLETE）
最もネガティブな店舗の状況をLLMに渡して、具体的な改善施策を生成させます。  
`COMPLETE()` の第一引数はモデル名。`mistral-large2`、`llama3.1-70b`、`claude-3-5-sonnet` 等から選択できます。


In [ ]:
-- 最もネガティブな店舗の改善施策をLLMが提案
WITH WORST_STORE AS (
    SELECT
        st.STORE_NAME,
        st.PREFECTURE,
        st.CITY,
        am.AREA_TYPE,
        cr.STORE_ID,
        ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(cr.REVIEW_TEXT)), 3)      AS AVG_SENTIMENT,
        COUNT(cr.REVIEW_ID)                                            AS REVIEW_COUNT
    FROM CUSTOMER_REVIEWS cr
    JOIN SNOWMART_STORES st ON cr.STORE_ID = st.STORE_ID
    JOIN AREA_MASTER am ON st.PREFECTURE = am.PREFECTURE AND st.CITY = am.CITY
    WHERE cr.STORE_ID = (
        SELECT STORE_ID FROM CUSTOMER_REVIEWS
        GROUP BY STORE_ID HAVING COUNT(*) >= 3
        ORDER BY AVG(SNOWFLAKE.CORTEX.SENTIMENT(REVIEW_TEXT)) ASC
        LIMIT 1
    )
    GROUP BY st.STORE_NAME, st.PREFECTURE, st.CITY, am.AREA_TYPE, cr.STORE_ID
)
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'mistral-large2',
    CONCAT(
        '以下のコンビニ店舗データに基づき、改善施策を3つ、具体的かつ簡潔に提案してください。\n\n',
        '店舗名: ', STORE_NAME, '\n',
        '所在地: ', PREFECTURE, ' ', CITY, '（', AREA_TYPE, '）\n',
        '平均日販: ', (SELECT ROUND(AVG(SALES_AMOUNT)) FROM SNOWMART_DB.SNOWMART_SCHEMA.DAILY_SALES WHERE STORE_ID = WORST_STORE.STORE_ID)::VARCHAR, '円\n',
        '顧客感情スコア: ', AVG_SENTIMENT::VARCHAR, '（-1が最悪、+1が最良）\n',
        'レビュー件数: ', REVIEW_COUNT::VARCHAR, '件\n\n',
        '改善施策（各50字以内で）:'
    )
) AS IMPROVEMENT_SUGGESTIONS
FROM WORST_STORE;

### Scene 1 ポイント
- `SENTIMENT` / `SUMMARIZE` / `CLASSIFY_TEXT` / `COMPLETE` はすべて `SELECT` の中に書くだけ
- `COMPLETE()` のモデルは `mistral-large2`、`llama3.1-70b`、`claude-3-5-sonnet` 等から選択可能
- データが Snowflake 外に出ないため、顧客レビューのような個人情報を含むデータも安全に使えます
- **AWS比較**: Amazon Comprehend + Bedrock API + Lambda の組み合わせ → Snowflake では SQL 一本


---
## Scene 2: Cortex Search — 社内ガイドラインを検索する



### Step 2-1: Cortex Search Service の作成
ドキュメントテーブルに対して Cortex Search Service を作成します。  
インデックスが自動構築されるので、次のセルから自然言語検索が使えるようになります。


In [ ]:
-- Cortex Search Service を作成（インデックス構築に1〜2分かかります）
CREATE OR REPLACE CORTEX SEARCH SERVICE SNOWMART_DOC_SEARCH
    ON DOC_TEXT
    ATTRIBUTES DOC_TITLE, DOC_SECTION
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 hour'
    AS (
        SELECT DOC_ID, DOC_TITLE, DOC_SECTION, DOC_TEXT
        FROM STORE_DOCUMENTS
    );

In [ ]:
-- 作成した Cortex Search Service を確認
SHOW CORTEX SEARCH SERVICES IN SCHEMA SNOWMART_DB.SNOWMART_SCHEMA;

-- 登録済みドキュメントの一覧
SELECT DOC_ID, DOC_SECTION FROM STORE_DOCUMENTS ORDER BY DOC_ID;

In [ ]:
-- 「住宅地への出店条件は？」
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWMART_DB.SNOWMART_SCHEMA.SNOWMART_DOC_SEARCH',
        '{
            "query": "住宅地に出店する際の条件と注意点を教えてください",
            "columns": ["DOC_SECTION", "DOC_TEXT"],
            "limit": 3
        }'
    )
) AS SEARCH_RESULTS;

In [ ]:
-- 「競合が多いエリアではどうする？」
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWMART_DB.SNOWMART_SCHEMA.SNOWMART_DOC_SEARCH',
        '{
            "query": "競合コンビニが多いエリアへの出店はどう判断すればよいですか",
            "columns": ["DOC_SECTION", "DOC_TEXT"],
            "limit": 3
        }'
    )
) AS SEARCH_RESULTS;

In [ ]:
-- 「出店候補のスコアリング方法は？」
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'SNOWMART_DB.SNOWMART_SCHEMA.SNOWMART_DOC_SEARCH',
        '{
            "query": "出店候補エリアをどうやって評価・採点すればいいですか",
            "columns": ["DOC_SECTION", "DOC_TEXT"],
            "limit": 2
        }'
    )
) AS SEARCH_RESULTS;

### 補足: Snowsight GUI から Cortex Search を使う

SQL の `SEARCH_PREVIEW()` に加えて、Snowsight の GUI から直接テストできます。

**アクセス方法**:  
Snowsight 左メニュー → **AI & ML** → **Cortex Search**  
→ `SNOWMART_DB` → `SNOWMART_SCHEMA` → `SNOWMART_DOC_SEARCH` を選択

**GUI でできること**:
- クエリボックスに自然言語を入力してリアルタイム検索
- 検索結果（ドキュメントの該当箇所）をハイライト表示で確認
- Source columns の表示切り替え（DOC_SECTION / DOC_TEXT）
- SQL を書かずに検索精度を素早くテスト可能

**試してみる質問**:
- 「住宅地への出店条件は？」
- 「競合が多いエリアの対策は？」
- 「出店候補エリアの評価方法は？」


### Scene 2 ポイント
- テーブルに文書を入れて `CREATE CORTEX SEARCH SERVICE` するだけ。Embedding モデルの選定・管理・インデックス運用が不要
- **セマンティック検索**: キーワードと違う言葉で質問しても関連箇所が返ってくる
- **AWS比較**: Bedrock Knowledge Base + OpenSearch Serverless + Lambda → Snowflake SQL 数行で完結
- 次の Scene 4 でこの Search Service を Cortex Agent のツールとして組み込みます


---
## Scene 3: Semantic View + Cortex Analyst — 自然言語でデータ分析



### Step 3-1: Semantic View の作成
テーブルのカラムにビジネス的な意味とシノニム（別名）を定義します。  
`SYNONYMS` に日本語・英語の両方を定義しておくと、どちらで質問しても正しく解釈されます。


In [ ]:
-- 複数テーブルを結合してビジネス意味を定義する Semantic View
-- DAILY_SALES（売上）, SNOWMART_STORES（店舗）, AREA_MASTER（エリア）, COMPETITOR_STORES（競合）を統合
CREATE OR REPLACE SEMANTIC VIEW SNOWMART_ANALYSIS
    TABLES (
        sales      AS SNOWMART_DB.SNOWMART_SCHEMA.DAILY_SALES
                       PRIMARY KEY (STORE_ID, SALES_DATE),
        stores     AS SNOWMART_DB.SNOWMART_SCHEMA.SNOWMART_STORES
                       PRIMARY KEY (STORE_ID),
        area       AS SNOWMART_DB.SNOWMART_SCHEMA.AREA_MASTER
                       PRIMARY KEY (AREA_ID) UNIQUE (PREFECTURE, CITY),
        competitor AS SNOWMART_DB.SNOWMART_SCHEMA.COMPETITOR_STORES
                       PRIMARY KEY (COMPETITOR_ID)
    )
    RELATIONSHIPS (
        sales (STORE_ID) REFERENCES stores,
        stores (PREFECTURE, CITY) REFERENCES area (PREFECTURE, CITY),
        competitor (PREFECTURE, CITY) REFERENCES area (PREFECTURE, CITY)
    )
    DIMENSIONS (
        sales.store_id         AS STORE_ID         WITH SYNONYMS = ('店舗ID'),
        stores.store_name      AS STORE_NAME       WITH SYNONYMS = ('店舗名', '店名'),
        stores.prefecture      AS PREFECTURE       WITH SYNONYMS = ('都道府県', '県'),
        stores.city            AS CITY             WITH SYNONYMS = ('市区町村', '市', '区'),
        stores.store_type      AS STORE_TYPE       WITH SYNONYMS = ('店舗種別', '直営FC'),
        area.area_type         AS AREA_TYPE        WITH SYNONYMS = ('エリア種別', '立地タイプ'),
        stores.nearest_station AS NEAREST_STATION  WITH SYNONYMS = ('最寄り駅', '駅'),
        stores.floor_area_sqm  AS FLOOR_AREA_SQM       WITH SYNONYMS = ('店舗面積', '売場面積', '面積'),
        sales.sales_date       AS SALES_DATE       WITH SYNONYMS = ('売上日', '日付')
    )
    METRICS (
        sales.avg_daily_sales    AS AVG(sales.SALES_AMOUNT)                  WITH SYNONYMS = ('日次売上', '平均日販', '日販', '売上'),
        sales.total_customers    AS SUM(sales.CUSTOMER_COUNT)                WITH SYNONYMS = ('来客数', '顧客数'),
        sales.avg_unit_price     AS AVG(sales.AVG_UNIT_PRICE)                WITH SYNONYMS = ('客単価', '平均単価'),
        sales.food_sales         AS SUM(sales.FOOD_SALES)                    WITH SYNONYMS = ('フード売上', '食品売上'),
        sales.beverage_sales     AS SUM(sales.BEVERAGE_SALES)                WITH SYNONYMS = ('飲料売上'),
        sales.daily_goods_sales  AS SUM(sales.DAILY_GOODS_SALES)             WITH SYNONYMS = ('日用品売上'),
        area.avg_population      AS AVG(area.POPULATION)                     WITH SYNONYMS = ('人口', '商圏人口'),
        area.avg_income          AS AVG(area.AVG_ANNUAL_INCOME)              WITH SYNONYMS = ('平均年収', '所得水準', '収入'),
        area.avg_households      AS AVG(area.HOUSEHOLDS)                     WITH SYNONYMS = ('世帯数', '住宅世帯'),
        area.avg_daytime_ratio   AS AVG(area.DAYTIME_POPULATION_RATIO)       WITH SYNONYMS = ('昼間人口比率', '昼夜比', 'オフィス街指標'),
        competitor.competitor_count AS COUNT(DISTINCT competitor.COMPETITOR_ID) WITH SYNONYMS = ('競合店数', '競合コンビニ数', '競合')
    );

In [ ]:
-- Semantic View の定義を確認
DESCRIBE SEMANTIC VIEW SNOWMART_ANALYSIS;

### 補足: Snowsight GUI で Semantic View を確認・管理する

**アクセス方法（データブラウザ）**:  
Snowsight 左メニュー → **カタログ** → データベースエクスプローラー  
→ `SNOWMART_DB` → `SNOWMART_SCHEMA` → **Semantic Views** → `SNOWMART_ANALYSIS`

**GUI で確認できる内容**:
- DIMENSIONS 一覧（カラム名・型 設定）
- METRICS 一覧（集計定義 設定）
- Relationtip の定義
- 「Cortex Analyst で開く」ボタンで Cortex Analyst に直接リンク

**SYNONYMS の重要性**:  
`SYNONYMS = ('日次売上', '平均日販', '日販', '売上')` のように登録しておくと、  日本語の質問でも正しくメトリクスが解釈されます。

> **補足**: YAML エディタで `SNOWMART_ANALYSIS` を開き、`avg_daily_sales` の定義と SYNONYMS を確認することができます。


### Step 3-2: Cortex Analyst で自然言語クエリ

**操作手順**:  
Snowsight 左メニュー → **AI & ML** → **Cortex Analyst**  
セマンティックビュー で `SNOWMART_DB.SNOWMART_SCHEMA.SNOWMART_ANALYSIS` を選択 → 右側のサイドパネルで「プレイグラウンド」を選択

まず、真ん中に表示されている「データセットを説明」を押すことでこのセマンティックビューで取り扱っているデータセットの説明が表示されます。

**試してみる質問例（順番に入力してください）**:

**Q1**: 「都道府県別の平均日販を高い順に教えてください」  
→ 都道府県別 `AVG_DAILY_SALES` のSQLが自動生成されます

**Q2**: 「競合店数が少なく人口が多いエリアのTOP10を教えてください」  
→ 出店チャンスのあるエリアの候補を検索します

**Q3**: 「住宅地エリアと商業地エリアの平均来客数を比較してください」  
→ `AREA_TYPE` でフィルタした比較集計を自動生成

**Q4**: 「昼間人口比率が高いエリアほど飲料売上が多い傾向はありますか？」  

> **ポイント**: 各質問の答えとセットで、セマンティッククエリが表示されます。 
> この SQL は「確認済みクエリ」として、セマンティックビューに登録することが可能です。確認済みクエリを登録することで、精度向上やレイテンシ低減につなげることができます。


### Scene 3 ポイント
- `SYNONYMS` に日本語を定義しておくと、日本語の質問でも正しく解釈される
- プレイグラウンド上で生成された SQL は直接確認・コピー可能。「確認済みクエリ」として登録することで、精度向上やレイテンシ低減につなげることができる


---
## Scene 4: Cortex Agent — AIエージェントで出店先を決める



### Step 4-1: Cortex Agent の作成
2つのツールを登録します:
- **Analyst**: Semantic View を使ってデータ分析（Scene 3 で作成したもの）
- **Search**: Cortex Search Service でガイドライン検索（Scene 2 で確認したもの）

`DESCRIPTION` が重要です。エージェントはここを読んで、どのツールを使うか判断します。


In [ ]:
CREATE OR REPLACE AGENT SNOWMART_AGENT
    COMMENT = 'スノーマート出店戦略AIエージェント'
    FROM SPECIFICATION $$
        instructions:
          system: 'あなたはスノーマート（全国500店舗のコンビニチェーン）の出店戦略アナリストAIです。データ分析が必要な質問は Analyst ツールを、方針・ガイドラインの質問は Search ツールを使ってください。複合的な判断が必要な場合は両方のツールを組み合わせてください。回答は日本語で、具体的な数値や根拠を含めてください。'
        tools:
          - tool_spec:
              type: 'cortex_analyst_text_to_sql'
              name: 'Analyst'
              description: 'スノーマートの店舗売上・エリア・来客数などの定量的なデータ分析に使います。'
          - tool_spec:
              type: 'cortex_search'
              name: 'Search'
              description: '出店ガイドライン・社内方針などの文書を検索します。立地条件・競合対策・売上目標などの方針的な質問に使います。'
        tool_resources:
          Analyst:
            semantic_view: 'SNOWMART_DB.SNOWMART_SCHEMA.SNOWMART_ANALYSIS'
          Search:
            name: 'SNOWMART_DB.SNOWMART_SCHEMA.SNOWMART_DOC_SEARCH'
            max_results: '3'
    $$;

### 補足: Snowsight GUI から Cortex Agent を作成・管理する

SQL の `CREATE AGENT` に加えて、Snowsight の GUI ウィザードでもエージェントを作成できます。

**アクセス方法（エージェント管理）**:  
Snowsight 左メニュー → **AI & ML** → **Agents**  
→ `SNOWMART_AGENT` が作成済みで表示されています

**GUI ウィザードで新規作成する場合**（`+ New Agent`）:
1. エージェント名・コメントを入力
2. **System Prompt** にエージェントへの指示を記述
3. **Add Tool** → ツール種別を選択
   - `Cortex Analyst`: Semantic View を選択（例: `SNOWMART_ANALYSIS`）
   - `Cortex Search`: Search Service を選択（例: `SNOWMART_DOC_SEARCH`）
4. 各ツールの **Description** を記述（エージェントがどのツールを使うか判断する根拠）
5. **Create** → すぐに Intelligence でテスト可能

**作成後の動作確認**:  
エージェント一覧で `SNOWMART_AGENT` を選択 → **「Test in Intelligence」** ボタンをクリック  
→ 次の Step 4-2 と同じ画面が開きます

> **ポイント**: Snowflake Intelligence で正式に利用するためには、エージェント画面で、「エージェントを追加」 のボタンを押す必要があります。


### Step 4-2: Snowflake Intelligence でエージェントと対話

**操作手順**:  
Snowsight 左メニュー → **AI & ML** → **Snowflake Intelligence**  
エージェント `SNOWMART_AGENT` を選択

**質問を順番に入力してください（各質問でどのツールが使われたか確認しましょう）**:

---

**質問 1（ウォームアップ）**:  
```
売上が最も高い都道府県TOP5と、その平均日販を教えてください
```
→ Analyst ツールだけを使います。Thinking プロセスで確認してみましょう。

---

**質問 2（出店チャンスを探す）**:  
```
競合店舗数が少なく、人口が多いエリアはどこですか？出店チャンスが高そうなエリアをTOP5で教えてください
```
→ Analyst ツールで競合数・人口データを分析します。

---

**質問 3（複数ツールを組み合わせた検索）**:  
```
先ほどの候補エリアについて、出店ガイドラインの条件を満たしているか確認してください
```
→ エージェントが Analyst ツール（データ）と Search ツール（ガイドライン）を **自律的に組み合わせて** 回答します。  
→ 「両方のツールを使った」ことを Thinking プロセスで確認してください。

---

**質問 4（リスク確認）**:  
```
候補エリアで注意すべき回避条件や競合チェーンの特徴をガイドラインから教えてください
```
→ Search ツールが競合対策・回避条件の文書を検索します。

---

**質問 5（最終提案）**:  
```
以上の分析をまとめて、次の出店先として最もお勧めのエリアを1つ挙げ、データによる根拠とガイドラインへの適合性の両方から理由を説明してください
```
→ データ ＋ ガイドライン を統合した最終提案が生成されます。**今日のゴールです。**


### Scene 4 ポイント
- エージェントは質問の内容から「データが必要か」「文書が必要か」を **自律的に判断** します
- **Thinking プロセス** で AI の推論過程が可視化されます。なぜその答えを出したか追跡することが可能です
- Cortex Agents は REST API でも呼び出せるので、社内 Web アプリや Slack Bot への組み込みも可能


---
## まとめ

### 今日体験した機能と AWS 対比

**Cortex AI Functions**（Scene 1）
- `SENTIMENT()` — 感情スコア → AWS Amazon Comprehend 相当
- `SUMMARIZE()` — 長文要約 → Bedrock + Lambda 相当
- `CLASSIFY_TEXT()` — テキスト分類 → Comprehend Custom Classifier 相当
- `COMPLETE()` — LLM呼び出し・テキスト生成 → Bedrock API 相当
- 共通の強み: **SQL一行、APIキー不要、データ外部流出なし**

**Cortex Search**（Scene 2）
- 文書をテーブルに入れるだけで RAG 基盤が完成
- AWS: Bedrock Knowledge Base + OpenSearch Serverless 相当

**Semantic View + Cortex Analyst**（Scene 3）
- 自然言語 → SQL自動生成でデータ分析

**Cortex Agent**（Scene 4）
- 複数AIツールを自律的にオーケストレーション

### すべてSnowflakeの中で完結する理由
- データが外部に出ない（セキュリティ・コンプライアンス）
- SQL と Python だけで書ける（既存スキルがそのまま使える）
- インフラ管理ゼロ（デプロイ不要）
- データ基盤 → AI分析 → RAG → エージェント → ダッシュボードまで1プラットフォーム

### 次に試すとよい機能
- `AI_EXTRACT()` — 非構造化テキストから構造化データを抽出
- `TRANSLATE()` — 多言語対応（英語レビューの日本語変換等）
- **Snowflake Notebooks** — Jupyter互換ノートブックで Snowpark Python + AI を組み合わせ
- **Cortex Fine-tuning** — 自社データでLLMをファインチューニング


---
## Scene 5: Streamlit ダッシュボード [オプション]


### 作成手順
1. Snowsight 左メニュー → **Projects** → **Streamlit** → **+ Streamlit App**
2. 設定: App name=`SNOWMART_DASHBOARD` / Database=`SNOWMART_DB` / Schema=`SNOWMART_SCHEMA` / Warehouse=`COMPUTE_WH`
3. 「Create」後、次のセルのコードをエディタに貼り付けて **Run**


In [ ]:
# === このコードを Streamlit in Snowflake のエディタに貼り付けてください ===
import streamlit as st
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title("スノーマート 出店戦略ダッシュボード")

tab1, tab2, tab3 = st.tabs(["エリア別売上", "AI レビュー分析", "出店チャンス"])

# ── Tab 1: エリア別売上 ──────────────────────────────────
with tab1:
    st.subheader("都道府県別 平均日次売上")
    df = session.sql("""
        SELECT st.PREFECTURE,
               ROUND(AVG(ds.SALES_AMOUNT)) AS AVG_DAILY_SALES,
               COUNT(DISTINCT ds.STORE_ID) AS STORES
        FROM SNOWMART_DB.SNOWMART_SCHEMA.DAILY_SALES ds
        JOIN SNOWMART_DB.SNOWMART_SCHEMA.SNOWMART_STORES st ON ds.STORE_ID = st.STORE_ID
        GROUP BY st.PREFECTURE ORDER BY AVG_DAILY_SALES DESC LIMIT 15
    """).to_pandas()
    kpi1, kpi2 = st.columns(2)
    kpi1.metric("全国平均日販", f"¥{int(df['AVG_DAILY_SALES'].mean()):,}")
    kpi2.metric("集計店舗数",   f"{df['STORES'].sum()} 店")
    st.bar_chart(df.set_index("PREFECTURE")["AVG_DAILY_SALES"])
    st.dataframe(df, use_container_width=True)

# ── Tab 2: AI レビュー分析 ───────────────────────────────
with tab2:
    st.subheader("Cortex AI による感情スコア分析")
    if st.button("感情分析を実行（約30秒）"):
        with st.spinner("Cortex AI Functions で分析中..."):
            df2 = session.sql("""
                SELECT cr.STORE_ID, st.STORE_NAME, st.PREFECTURE,
                       COUNT(*) AS CNT,
                       ROUND(AVG(cr.RATING), 1) AS AVG_RATING,
                       ROUND(AVG(SNOWFLAKE.CORTEX.SENTIMENT(cr.REVIEW_TEXT)), 3) AS AVG_SENTIMENT
                FROM CUSTOMER_REVIEWS cr
                JOIN SNOWMART_STORES st ON cr.STORE_ID = st.STORE_ID
                GROUP BY cr.STORE_ID, st.STORE_NAME, st.PREFECTURE
                HAVING COUNT(*) >= 3 ORDER BY AVG_SENTIMENT ASC LIMIT 15
            """).to_pandas()
        st.bar_chart(df2.set_index("STORE_NAME")["AVG_SENTIMENT"])
        st.dataframe(df2, use_container_width=True)

# ── Tab 3: 出店チャンス ──────────────────────────────────
with tab3:
    st.subheader("出店チャンスエリア（競合少 × 人口多）")
    df3 = session.sql("""
        SELECT a.PREFECTURE, a.CITY, a.AREA_TYPE, a.POPULATION,
               COALESCE(c.CNT, 0) AS COMPETITOR_COUNT,
               ROUND(a.POPULATION / NULLIF(COALESCE(c.CNT, 0) + 1, 0)) AS SCORE
        FROM AREA_MASTER a
        LEFT JOIN (SELECT PREFECTURE, CITY, COUNT(*) AS CNT
                   FROM COMPETITOR_STORES GROUP BY PREFECTURE, CITY) c
          ON a.PREFECTURE = c.PREFECTURE AND a.CITY = c.CITY
        WHERE COALESCE(c.CNT, 0) <= 3
        ORDER BY SCORE DESC LIMIT 10
    """).to_pandas()
    if not df3.empty:
        top = df3.iloc[0]
        st.success(f"最有力候補: **{top['PREFECTURE']} {top['CITY']}** "
                   f"（{top['AREA_TYPE']} / 競合{int(top['COMPETITOR_COUNT'])}店 / 人口{int(top['POPULATION']):,}人）")
    st.dataframe(df3, use_container_width=True)

st.caption("Powered by Snowflake Cortex AI | SnowMart Data Team")